In [0]:
# imports
import requests
import json
from datetime import datetime, timedelta

In [0]:
dbutils.widgets.text("API_KEY",dbutils.secrets.get(scope='Connectors', key='api-key'))
dbutils.widgets.text("SAS_TOKEN",dbutils.secrets.get(scope='Connectors', key='SAS-token'))

In [0]:
spark.conf.set("fs.azure.account.auth.type.bitcoindatalake.dfs.core.windows.net", "SAS")
spark.conf.set("fs.azure.sas.token.provider.type.bitcoindatalake.dfs.core.windows.net", "org.apache.hadoop.fs.azurebfs.sas.FixedSASTokenProvider")
spark.conf.set("fs.azure.sas.fixed.token.bitcoindatalake.dfs.core.windows.net", dbutils.widgets.get("SAS_TOKEN"))

In [0]:
# Check if connection is set up
dbutils.fs.ls("abfss://bronze@bitcoindatalake.dfs.core.windows.net/")

In [0]:
# API call to get the historical chart data of bitcoin per hour within the past 90 days
# json response example:
# {
#   "prices": [
#     [
#       1704067241331, represent the time in UNIX format
#       42261.0406175669
#     ],
#     [
#       1704070847420,
#       42493.2764087546
#     ],
#     [
#       1704074443652,
#       42654.0731066594
#     ]
#   ],
#   "market_caps": [
#     [
#       1704067241331,
#       827596236151.196
#     ],
#     [
#       1704070847420,
#       831531023621.411
#     ],
#     [
#       1704074443652,
#       835499399014.932
#     ]
#   ],
#   "total_volumes": [
#     [
#       1704067241331,
#       14305769170.9498
#     ],
#     [
#       1704070847420,
#       14130205376.1709
#     ],
#     [
#       1704074443652,
#       13697382902.2424
#     ]
#   ]
# }

In [0]:

url = "https://api.coingecko.com/api/v3/coins/bitcoin/market_chart/range"
end_time = datetime.now()
start_time = end_time - timedelta(days=90)
from_timestamp = start_time.timestamp()
to_timestamp = end_time.timestamp()
params = {
    "vs_currency": "usd",
    "from": from_timestamp,
    "to": to_timestamp
}
headers = {
    "x-cg-demo-api-key" : dbutils.widgets.get("API_KEY")
}
response = requests.get(url, headers=headers, params=params)

In [0]:
if response.status_code == 200:
         chart_data_last90days = response.json()
else:
    print("the API call failed")

In [0]:
# API call to get the OHLC chart (Open, High, Low, Close) of bitcoin within a the last 90 days
# json response example:
# [
#   [
#     1709395200000,
#     61942,
#     62211,
#     61721,
#     61845
#   ],
#   [
#     1709409600000,
#     61828,
#     62139,
#     61726,
#     62139
#   ],
#   [
#     1709424000000,
#     62171,
#     62210,
#     61821,
#     62068
#   ]
# ]

In [0]:
url = "https://api.coingecko.com/api/v3/coins/bitcoin/ohlc?vs_currency=usd"
params_lastday  = {
    "vs_currency": "usd",
    "days": "1"
}
params_last30days  = {
    "vs_currency": "usd",
    "days": "30"
}
headers = {
    "x-cg-demo-api-key" : dbutils.widgets.get("API_KEY")
}
response_lastday = requests.get(url, headers=headers, params=params_lastday )
response_last30days = requests.get(url, headers=headers, params=params_last30days)
if response_lastday.status_code == 200 and response_last30days.status_code == 200:
        ohlc_data_lastday = response_lastday.json()
        ohlc_data_last30days = response_last30days.json()
else:
    print(f"data Last day{response_lastday}")
    print(f"data Last 30 days{response_last30days}")

In [0]:
# Upload json data:
# response_lastday
# response_last30days
# chart_data_last90days
# to raw container